### Task 02  ·  API → Clean → Save 

In [1]:
import pandas as pd
import requests
import mysql.connector

In [2]:
response = requests.get('https://jsonplaceholder.typicode.com/posts')
data = response.json()
data[0:2]

[{'userId': 1,
  'id': 1,
  'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
  'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'},
 {'userId': 1,
  'id': 2,
  'title': 'qui est esse',
  'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}]

In [3]:
df = pd.DataFrame(data)
df.head()

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [4]:
# Adding a word_count  column ———> which count words in the title
df['word_count'] = df['title'].str.split().str.len()
df.head()

,userId,id,title,body,word_count
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...,9
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...,3
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...,3


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   userId      100 non-null    int64
 1   id          100 non-null    int64
 2   title       100 non-null    str  
 3   body        100 non-null    str  
 4   word_count  100 non-null    int64
dtypes: int64(3), str(2)
memory usage: 4.0 KB


In [6]:
#  Keeping only posts where word_count >= 4
filtered_df = df[df['word_count'] >= 4]
filtered_df.head()

,userId,id,title,body,word_count
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...,9
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4
5,1,6,dolorem eum magni eos aperiam quia,ut aspernatur corporis harum nihil quis provid...,6
7,1,8,dolorem dolore est ipsam,dignissimos aperiam dolorem qui eum\nfacilis q...,4


In [7]:
# Standardise: title to title case, strip whitespace from body
filtered_df['title'] = filtered_df['title'].str.title()
filtered_df['body'] = filtered_df['body'].str.strip()
filtered_df.head()

,userId,id,title,body,word_count
0,1,1,Sunt Aut Facere Repellat Provident Occaecati E...,quia et suscipit\nsuscipit recusandae consequu...,9
2,1,3,Ea Molestias Quasi Exercitationem Repellat Qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9
3,1,4,Eum Et Est Occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4
5,1,6,Dolorem Eum Magni Eos Aperiam Quia,ut aspernatur corporis harum nihil quis provid...,6
7,1,8,Dolorem Dolore Est Ipsam,dignissimos aperiam dolorem qui eum\nfacilis q...,4


In [8]:
# LOAD: Save cleaned data to  clean_posts.csv  (no index) and to mysql  posts.db  using df.to_sql()
filtered_df.to_csv('clean_posts.csv', index=False)

# Save to MySQL
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='root',
)
cursor = conn.cursor()
cursor.execute('CREATE DATABASE IF NOT EXISTS posts_db')
cursor.execute('USE posts_db')
cursor.execute('''  
    CREATE TABLE IF NOT EXISTS posts (
        id INT PRIMARY KEY,
        userId INT,
        title VARCHAR(255),
        body TEXT,
        word_count INT
)''')

for index, row in filtered_df.iterrows():
    cursor.execute('''  
        INSERT IGNORE INTO posts (id, userId, title, body, word_count) 
        VALUES (%s, %s, %s, %s, %s)''', 
        (row['id'], row['userId'], row['title'], row['body'], row['word_count'])
    )
conn.commit()
if conn.is_connected():
    cursor.close()
    conn.close()

In [9]:
# Print: total posts fetched, posts after filter, and top 3 users by post count.
print(f'Total posts fetched: {len(df)}')
print(f'Posts after filter: {len(filtered_df)}')
top_users = filtered_df['userId'].value_counts().head(3)
print('Top 3 users by post count:')
print(top_users)

Total posts fetched: 100
Posts after filter: 89
Top 3 users by post count:
userId
3    10
6    10
8    10
Name: count, dtype: int64
